In [1]:
#!/usr/bin/env python3
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   ConvNeXt V1 Base — Dark Matter Substructure Classification                ║
║   Kaggle H100 · 5 (S1 head) + 40 (S2 LLRD) + 40 (S3 extension) = 85 ep    ║
║   Metric : Macro OvR ROC-AUC                                                ║
║                                                                              ║
║   Differences vs ConvNeXt V2 Large script:                                  ║
║     1. MODEL_NAME  → convnext_base.fb_in22k_ft_in1k  (88.6M, supervised)   ║
║     2. BATCH_SIZE  → 256  (88M params fits; doubles effective LR)           ║
║     3. BASE_LR     → 6.25e-4  (effective = 6.25e-4 × 256/256 = 6.25e-4)   ║
║     4. STAGE1_EPOCHS → 5  (supervised pretrain more robust to cold-start)   ║
║     5. S3_LR       → 6.25e-5  (10× lower than Stage 2 effective)           ║
║     6. no_wd set   → 'grn' removed  (V1 has no GRN layer)                  ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os, math, random, gc, warnings
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import timm
from timm.utils import ModelEma

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
class CFG:
    # ── Paths ──────────────────────────────────────────────────────────────────
    DATA_ROOT  = "/kaggle/input/datasets/stellarquant/deeplensetask1/dataset"
    OUTPUT_DIR = "/kaggle/working/convnextv1_base"

    # ── Model ──────────────────────────────────────────────────────────────────
    # V1 Base: 88.6M params, supervised IN-22K → IN-1K fine-tune
    # No GRN, no FCMAE — direct comparison point for V2 Large
    MODEL_NAME      = "convnext_base.fb_in22k_ft_in1k"
    NUM_CLASSES     = 3
    IMG_SIZE        = 224
    DROP_PATH       = 0.1
    HEAD_INIT_SCALE = 0.001

    # ── Stage epochs ───────────────────────────────────────────────────────────
    # Stage 1 is 5 epochs (not 10) — supervised pretraining is more robust
    # to cold-start gradients than FCMAE, so the head needs less warmup
    STAGE1_EPOCHS = 5
    STAGE2_EPOCHS = 40
    STAGE3_EPOCHS = 40

    # ── DataLoader ─────────────────────────────────────────────────────────────
    # 256 fits on H100 with 88M params + EMA + AdamW (unlike 196M V2 Large)
    BATCH_SIZE   = 256
    NUM_WORKERS  = 4
    VAL_SPLIT    = 0.10

    # ── Optimiser ──────────────────────────────────────────────────────────────
    # effective_lr = BASE_LR × BATCH_SIZE / 256 = 6.25e-4 × 1.0 = 6.25e-4
    BASE_LR      = 6.25e-4
    WEIGHT_DECAY = 0.05
    LAYER_DECAY  = 0.7
    MIN_LR       = 1e-6

    # ── Stage 3 LR ─────────────────────────────────────────────────────────────
    S3_LR        = 6.25e-5   # 10× lower than Stage 2 effective LR

    # ── Scheduler ─────────────────────────────────────────────────────────────
    S1_WARMUP = 1   # 1-epoch warmup sufficient for shorter Stage 1
    S2_WARMUP = 5

    # ── Regularisation ────────────────────────────────────────────────────────
    LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA     = 0.4

    # ── EMA ───────────────────────────────────────────────────────────────────
    EMA_DECAY = 0.9999

    # ── Precision & stability ─────────────────────────────────────────────────
    USE_BF16  = True
    GRAD_CLIP = 1.0

    # ── Dataset metadata ──────────────────────────────────────────────────────
    CLASS_NAMES = ['no_sub', 'subhalo', 'vortex']
    CLASS_DIRS  = {'no_sub': 'no', 'subhalo': 'sphere', 'vortex': 'vort'}
    PIXEL_MEAN  = 0.0615
    PIXEL_STD   = 0.1152
    SEED        = 42


# ══════════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════════
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════
class LensDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels     = labels
        self.transform  = transform

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        img = np.load(self.file_paths[idx]).astype(np.float32)
        if img.ndim == 3: img = img[0]
        img_u8 = (img * 255).clip(0, 255).astype(np.uint8)
        if self.transform:
            return self.transform(image=img_u8)['image'], self.labels[idx]
        return torch.from_numpy(img_u8[None]).float() / 255.0, self.labels[idx]


def auto_discover_root(base='/kaggle/input') -> Path:
    expected = set(CFG.CLASS_DIRS.values())
    for dirpath, dirnames, _ in os.walk(base):
        p = Path(dirpath)
        if {'train', 'val'}.issubset(set(dirnames)):
            train_p = p / 'train'
            if train_p.exists():
                found = {d.name for d in train_p.iterdir() if d.is_dir()}
                if expected & found:
                    print(f"  ✓ Auto-discovered DATA_ROOT = {p}")
                    return p
    raise FileNotFoundError(
        "Could not auto-discover dataset. "
        "Run: [print(r) for r,d,f in os.walk('/kaggle/input')]"
    )


def build_file_list(root: Path, split: str):
    paths, labels = [], []
    split_dir = root / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Split dir missing: {split_dir}")
    for i, cls in enumerate(CFG.CLASS_NAMES):
        cls_dir   = split_dir / CFG.CLASS_DIRS[cls]
        if not cls_dir.exists():
            raise FileNotFoundError(f"Class dir missing: {cls_dir}")
        npy_files = sorted(cls_dir.glob("*.npy")) + sorted(cls_dir.glob("*.NPY"))
        if not npy_files:
            raise FileNotFoundError(f"No .npy files in {cls_dir}")
        paths.extend(str(p) for p in npy_files)
        labels.extend([i] * len(npy_files))
        print(f"  [{split}/{cls}]  {len(npy_files):,} images  ← {cls_dir}")
    return paths, labels


# ══════════════════════════════════════════════════════════════════════════════
# AUGMENTATIONS  (identical to V2 — fair comparison requires same pipeline)
# ══════════════════════════════════════════════════════════════════════════════
def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=0.9, border_mode=0, value=0),
        A.RandomResizedCrop(
            size=(CFG.IMG_SIZE, CFG.IMG_SIZE),
            scale=(0.90, 1.00), ratio=(0.95, 1.05),
            interpolation=2, p=0.5,
        ),
        A.GaussNoise(var_limit=(0.65, 2.60), p=0.35),
        A.CoarseDropout(
            max_holes=4, max_height=18, max_width=18,
            min_holes=1, min_height=8,  min_width=8,
            fill_value=0, p=0.20,
        ),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def replicate_channels(x): return x.repeat(1, 3, 1, 1)

def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════
def build_model() -> nn.Module:
    model = timm.create_model(
        CFG.MODEL_NAME, pretrained=True,
        num_classes=CFG.NUM_CLASSES,
        drop_path_rate=CFG.DROP_PATH,
        in_chans=3,
    )
    # Near-zero head init — same as V2 for consistency
    head = getattr(model, 'head', None)
    if head is not None:
        fc = getattr(head, 'fc', head if isinstance(head, nn.Linear) else None)
        if isinstance(fc, nn.Linear):
            nn.init.trunc_normal_(fc.weight, std=0.02 * CFG.HEAD_INIT_SCALE)
            nn.init.constant_(fc.bias, 0)
    # Gradient checkpointing — optional at 88M but keeps VRAM headroom
    model.set_grad_checkpointing(enable=True)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Model : {CFG.MODEL_NAME}")
    print(f"  Params: {total/1e6:.1f}M  |  grad_ckpt=ON")
    return model


# ══════════════════════════════════════════════════════════════════════════════
# LAYER-WISE LR DECAY  (LLRD)
# ConvNeXt V1 uses identical timm naming to V2: stem → stages.0-3 → head
# Only difference: 'grn' removed from no_wd set (V1 has no GRN layer)
# ══════════════════════════════════════════════════════════════════════════════
def get_llrd_param_groups(model: nn.Module, base_lr: float) -> list:
    d     = CFG.LAYER_DECAY
    # 'grn' intentionally absent — ConvNeXt V1 does not have GRN layers
    no_wd = {'bias', 'norm', 'bn', 'ln', 'gamma', 'beta', 'LayerNorm', 'scale'}

    layer_map = {
        'head':     1.0,
        'norm_pre': d ** 1,
        'stages.3': d ** 1,
        'stages.2': d ** 2,
        'stages.1': d ** 3,
        'stages.0': d ** 4,
        'stem':     d ** 5,
    }

    groups, assigned = [], set()

    def skip_wd(name): return any(k in name for k in no_wd)

    for prefix, scale in layer_map.items():
        wd_p, no_wd_p = [], []
        for name, param in model.named_parameters():
            if not param.requires_grad or name in assigned: continue
            if not name.startswith(prefix): continue
            assigned.add(name)
            (no_wd_p if skip_wd(name) else wd_p).append(param)
        if wd_p:
            groups.append({'params': wd_p,    'lr': base_lr * scale,
                           'weight_decay': CFG.WEIGHT_DECAY})
        if no_wd_p:
            groups.append({'params': no_wd_p, 'lr': base_lr * scale,
                           'weight_decay': 0.0})

    remaining = [(n, p) for n, p in model.named_parameters()
                 if p.requires_grad and n not in assigned]
    if remaining:
        groups.append({'params': [p for _, p in remaining],
                       'lr': base_lr * d**6, 'weight_decay': CFG.WEIGHT_DECAY})

    total_p = sum(p.numel() for g in groups for p in g['params'])
    print(f"  LLRD  : {len(groups)} groups | base_lr={base_lr:.2e} "
          f"| decay={d} | {total_p/1e6:.1f}M params")
    return groups


# ══════════════════════════════════════════════════════════════════════════════
# SCHEDULER
# ══════════════════════════════════════════════════════════════════════════════
def cosine_with_warmup(optimizer, warmup_epochs: int, total_epochs: int,
                        min_lr_ratio: float = 0.01):
    def _fn(ep):
        if ep < warmup_epochs:
            return max(1e-6, (ep + 1) / warmup_epochs)
        prog = (ep - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr_ratio + 0.5 * (1.0 - min_lr_ratio) * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, _fn)


# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    dtype = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc='Eval', leave=False):
        imgs = replicate_channels(imgs).to(device, non_blocking=True)
        with torch.autocast('cuda', dtype=dtype):
            probs = F.softmax(model(imgs), dim=-1)
        all_probs.append(probs.cpu().float().numpy())
        all_labels.append(labels.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    macro  = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    per_cls = {cls: roc_auc_score((labels==i).astype(int), probs[:,i])
               for i, cls in enumerate(CFG.CLASS_NAMES)}
    return macro, per_cls, probs, labels


# ══════════════════════════════════════════════════════════════════════════════
# TRAIN ONE EPOCH
# ══════════════════════════════════════════════════════════════════════════════
def train_one_epoch(model, loader, optimizer, criterion, scheduler,
                    ema, device, label: str) -> float:
    model.train()
    dtype  = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    losses = []
    pbar   = tqdm(loader, desc=label, leave=True)
    for imgs, labels in pbar:
        imgs   = replicate_channels(imgs).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG.MIXUP_ALPHA)
        with torch.autocast('cuda', dtype=dtype):
            logits = model(imgs)
            loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        optimizer.step()
        ema.update(model)
        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'lr':   f'{optimizer.param_groups[0]["lr"]:.2e}'})
    scheduler.step()
    return float(np.mean(losses))


# ══════════════════════════════════════════════════════════════════════════════
# PLOTS
# ══════════════════════════════════════════════════════════════════════════════
def plot_roc_curves(labels, probs, save_path, title="Test Set"):
    COLORS = ['#e74c3c', '#2ecc71', '#3498db']
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'ROC Curves — {title}', fontsize=14, fontweight='bold')
    ax, per_aucs = axes[0], []
    for i, (cls, col) in enumerate(zip(CFG.CLASS_NAMES, COLORS)):
        binary      = (labels == i).astype(int)
        fpr, tpr, _ = roc_curve(binary, probs[:, i])
        auc_val     = roc_auc_score(binary, probs[:, i])
        per_aucs.append(auc_val)
        ax.plot(fpr, tpr, color=col, lw=2.2, label=f'{cls}  (AUC = {auc_val:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('One-vs-Rest ROC per Class')
    ax.legend(loc='lower right'); ax.grid(alpha=0.25)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    macro = np.mean(per_aucs)
    ax2   = axes[1]
    bars  = ax2.bar(CFG.CLASS_NAMES, per_aucs, color=COLORS,
                    alpha=0.85, edgecolor='white', linewidth=1.2)
    ax2.axhline(macro, color='gold', lw=2, ls='--', label=f'Macro = {macro:.4f}')
    ax2.set_ylim(max(0.5, min(per_aucs)-0.03), 1.005)
    ax2.set_ylabel('AUC'); ax2.set_title('Per-Class AUC Summary')
    ax2.legend(); ax2.grid(axis='y', alpha=0.25)
    for bar, val in zip(bars, per_aucs):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ROC plot  → {save_path}")


def plot_history(history, save_path):
    total = len(history['train_loss'])
    eps   = range(1, total + 1)
    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax1.plot(eps, history['train_loss'], color='#e74c3c', lw=2, label='Train Loss')
    s1 = CFG.STAGE1_EPOCHS
    s2 = s1 + CFG.STAGE2_EPOCHS
    ax1.axvline(s1 + 0.5, color='gray',   ls=':', lw=1.5, label='S1→S2')
    ax1.axvline(s2 + 0.5, color='orange', ls=':', lw=1.5, label='S2→S3')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss', color='#e74c3c')
    ax2 = ax1.twinx()
    ax2.plot(eps, history['val_auc'], color='#2ecc71', lw=2,
             label='Val Macro AUC (EMA)')
    ax2.set_ylabel('AUC', color='#2ecc71')
    lines  = ax1.get_legend_handles_labels()
    lines2 = ax2.get_legend_handles_labels()
    ax1.legend(lines[0]+lines2[0], lines[1]+lines2[1], loc='center right')
    ax1.set_title('Training History — ConvNeXt V1 Base  (S1 + S2 + S3)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  History   → {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
def main():
    seed_everything(CFG.SEED)
    os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA not available.\n"
            "In Kaggle: Settings → Accelerator → GPU (H100), then restart kernel."
        )
    device = torch.device('cuda')

    # ── File lists ────────────────────────────────────────────────────────────
    print("── Loading file lists ──")
    data_root = Path(CFG.DATA_ROOT)
    if not data_root.exists():
        print(f"  ⚠ Configured path not found, running auto-discovery...")
        data_root = auto_discover_root()
    else:
        print(f"  ✓ DATA_ROOT = {data_root}")

    train_paths, train_labels = build_file_list(data_root, 'train')
    test_paths,  test_labels  = build_file_list(data_root, 'val')

    tr_paths, val_paths, tr_labels, val_labels = train_test_split(
        train_paths, train_labels,
        test_size=CFG.VAL_SPLIT, stratify=train_labels,
        random_state=CFG.SEED,
    )
    print(f"\n  Train : {len(tr_paths):,}  |  Val : {len(val_paths):,}  "
          f"|  Test : {len(test_paths):,}\n")

    # ── Loaders ───────────────────────────────────────────────────────────────
    train_loader = DataLoader(
        LensDataset(tr_paths,   tr_labels,   get_train_transforms()),
        batch_size=CFG.BATCH_SIZE, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=True,
        drop_last=True, persistent_workers=True,
    )
    val_loader = DataLoader(
        LensDataset(val_paths,  val_labels,  get_val_transforms()),
        batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )
    test_loader = DataLoader(
        LensDataset(test_paths, test_labels, get_val_transforms()),
        batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )

    # ── Model + EMA ───────────────────────────────────────────────────────────
    print("\n── Building model ──")
    model     = build_model().to(device)
    ema       = ModelEma(model, decay=CFG.EMA_DECAY, device=device)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
    ckpt_path = f"{CFG.OUTPUT_DIR}/best_model.pth"

    history      = {'train_loss': [], 'val_auc': []}
    best_val_auc = 0.0

    def _log(stage_label, ep, total, loss, macro, per_cls):
        cls_str = '  '.join(f'{k}={v:.4f}' for k, v in per_cls.items())
        print(f"  [{stage_label} {ep:02d}/{total}]  "
              f"loss={loss:.4f}  macro_auc={macro:.4f}  |  {cls_str}")

    def _maybe_save(macro, ep_abs):
        nonlocal best_val_auc
        if macro > best_val_auc:
            best_val_auc = macro
            torch.save({'epoch': ep_abs, 'model': ema.ema.state_dict(),
                        'val_auc': macro}, ckpt_path)
            print(f"  ✓  New best val AUC={macro:.4f}  → checkpoint saved")

    # ══════════════════════════════════════════════════════════════════════════
    # STAGE 1 — Head-only warm-up (backbone frozen)
    # 5 epochs (not 10) — V1 supervised pretraining is more robust to
    # cold-start gradients than V2 FCMAE, so head adapts faster
    # ══════════════════════════════════════════════════════════════════════════
    print(f"\n{'═'*62}")
    print(f"  STAGE 1 — Head-only ({CFG.STAGE1_EPOCHS} epochs, backbone frozen)")
    print(f"{'═'*62}")

    for name, p in model.named_parameters():
        p.requires_grad = ('head' in name)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params: {trainable:,}  (head only)")

    s1_opt   = AdamW([p for p in model.parameters() if p.requires_grad],
                     lr=1e-3, weight_decay=CFG.WEIGHT_DECAY)
    s1_sched = cosine_with_warmup(s1_opt, CFG.S1_WARMUP, CFG.STAGE1_EPOCHS)

    for ep in range(CFG.STAGE1_EPOCHS):
        loss = train_one_epoch(model, train_loader, s1_opt, criterion,
                               s1_sched, ema, device,
                               f'S1 {ep+1:02d}/{CFG.STAGE1_EPOCHS}')
        macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
        history['train_loss'].append(loss)
        history['val_auc'].append(macro)
        _log('S1', ep+1, CFG.STAGE1_EPOCHS, loss, macro, per_cls)
        _maybe_save(macro, ep)

    # ══════════════════════════════════════════════════════════════════════════
    # STAGE 2 — Full fine-tuning with LLRD
    # Effective LR = 6.25e-4 (double V2's 3.125e-4 due to batch=256 vs 128)
    # ══════════════════════════════════════════════════════════════════════════
    print(f"\n{'═'*62}")
    print(f"  STAGE 2 — Full LLRD fine-tuning ({CFG.STAGE2_EPOCHS} epochs)")
    print(f"{'═'*62}")

    del s1_opt, s1_sched
    gc.collect(); torch.cuda.empty_cache()
    print(f"  GPU after cache flush: "
          f"{torch.cuda.memory_allocated()/1e9:.1f} GB alloc / "
          f"{torch.cuda.memory_reserved()/1e9:.1f} GB reserved")

    for p in model.parameters(): p.requires_grad = True

    eff_lr    = CFG.BASE_LR * CFG.BATCH_SIZE / 256   # 6.25e-4
    s2_groups = get_llrd_param_groups(model, eff_lr)
    s2_opt    = AdamW(s2_groups, weight_decay=CFG.WEIGHT_DECAY)
    s2_sched  = cosine_with_warmup(
        s2_opt, CFG.S2_WARMUP, CFG.STAGE2_EPOCHS,
        min_lr_ratio=CFG.MIN_LR / eff_lr,
    )

    for ep in range(CFG.STAGE2_EPOCHS):
        abs_ep = CFG.STAGE1_EPOCHS + ep
        loss   = train_one_epoch(model, train_loader, s2_opt, criterion,
                                 s2_sched, ema, device,
                                 f'S2 {ep+1:02d}/{CFG.STAGE2_EPOCHS}')
        macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
        history['train_loss'].append(loss)
        history['val_auc'].append(macro)
        _log('S2', ep+1, CFG.STAGE2_EPOCHS, loss, macro, per_cls)
        _maybe_save(macro, abs_ep)

    # ══════════════════════════════════════════════════════════════════════════
    # STAGE 3 — Low-LR extension
    # ══════════════════════════════════════════════════════════════════════════
    print(f"\n{'═'*62}")
    print(f"  STAGE 3 — Extension ({CFG.STAGE3_EPOCHS} epochs, LR={CFG.S3_LR:.2e})")
    print(f"{'═'*62}")

    del s2_opt, s2_sched
    gc.collect(); torch.cuda.empty_cache()

    s3_groups = get_llrd_param_groups(model, CFG.S3_LR)
    s3_opt    = AdamW(s3_groups, weight_decay=CFG.WEIGHT_DECAY)
    s3_sched  = cosine_with_warmup(
        s3_opt, warmup_epochs=0, total_epochs=CFG.STAGE3_EPOCHS,
        min_lr_ratio=0.01,
    )

    for ep in range(CFG.STAGE3_EPOCHS):
        abs_ep = CFG.STAGE1_EPOCHS + CFG.STAGE2_EPOCHS + ep
        loss   = train_one_epoch(model, train_loader, s3_opt, criterion,
                                 s3_sched, ema, device,
                                 f'S3 {ep+1:02d}/{CFG.STAGE3_EPOCHS}')
        macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
        history['train_loss'].append(loss)
        history['val_auc'].append(macro)
        _log('S3', ep+1, CFG.STAGE3_EPOCHS, loss, macro, per_cls)
        _maybe_save(macro, abs_ep)

    # ══════════════════════════════════════════════════════════════════════════
    # FINAL TEST EVALUATION
    # ══════════════════════════════════════════════════════════════════════════
    print(f"\n{'═'*62}")
    print("  FINAL TEST EVALUATION (best EMA checkpoint)")
    print(f"{'═'*62}")

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])

    test_auc, test_per_cls, test_probs, test_lbl = evaluate(model, test_loader, device)

    print(f"\n  Macro OvR AUC  : {test_auc:.4f}")
    for cls, val in test_per_cls.items():
        print(f"    {cls:>10}  : {val:.4f}")

    plot_roc_curves(
        test_lbl, test_probs,
        save_path=f"{CFG.OUTPUT_DIR}/roc_curves_final.png",
        title=f"Test Set — ConvNeXt V1 Base  (Macro AUC = {test_auc:.4f})",
    )
    plot_history(history, save_path=f"{CFG.OUTPUT_DIR}/training_history.png")

    print(f"\n{'─'*62}")
    print(f"  Best val AUC  : {best_val_auc:.4f}")
    print(f"  Test AUC      : {test_auc:.4f}")
    print(f"  Checkpoint    : {ckpt_path}")
    print(f"  Outputs       : {CFG.OUTPUT_DIR}")
    print(f"{'─'*62}\n")


if __name__ == '__main__':
    main()

── Loading file lists ──
  ✓ DATA_ROOT = /kaggle/input/datasets/stellarquant/deeplensetask1/dataset
  [train/no_sub]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/no
  [train/subhalo]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/sphere
  [train/vortex]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/vort
  [val/no_sub]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/no
  [val/subhalo]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/sphere
  [val/vortex]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/vort

  Train : 27,000  |  Val : 3,000  |  Test : 7,500


── Building model ──


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

  Model : convnext_base.fb_in22k_ft_in1k
  Params: 87.6M  |  grad_ckpt=ON

══════════════════════════════════════════════════════════════
  STAGE 1 — Head-only (5 epochs, backbone frozen)
══════════════════════════════════════════════════════════════
  Trainable params: 5,123  (head only)


S1 01/5:   0%|          | 0/105 [00:00<?, ?it/s]

Eval:   0%|          | 0/6 [00:00<?, ?it/s]

...

S3 40/40:   0%|          | 0/105 [00:00<?, ?it/s]

Eval:   0%|          | 0/6 [00:00<?, ?it/s]

  [S3 40/40]  loss=0.7617  macro_auc=0.9670  |  no_sub=0.9773  subhalo=0.9614  vortex=0.9624
  ✓  New best val AUC=0.9670  → checkpoint saved

══════════════════════════════════════════════════════════════
  FINAL TEST EVALUATION (best EMA checkpoint)
══════════════════════════════════════════════════════════════


Eval:   0%|          | 0/15 [00:00<?, ?it/s]


  Macro OvR AUC  : 0.9697
        no_sub  : 0.9799
       subhalo  : 0.9650
        vortex  : 0.9642
  ROC plot  → /kaggle/working/convnextv1_base/roc_curves_final.png
  History   → /kaggle/working/convnextv1_base/training_history.png

──────────────────────────────────────────────────────────────
  Best val AUC  : 0.9670
  Test AUC      : 0.9697
  Checkpoint    : /kaggle/working/convnextv1_base/best_model.pth
  Outputs       : /kaggle/working/convnextv1_base
──────────────────────────────────────────────────────────────

